In [ ]:
# 过滤警告'num_nodes'警告
import warnings


warnings.filterwarnings('ignore', 
    message="Unable to accurately infer 'num_nodes'",
    category=UserWarning,
    module='torch_geometric.data.storage')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_add_pool
from torch_geometric.nn.norm import BatchNorm
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from kan import *
import optuna
from optuna.samplers import TPESampler
from datetime import datetime


device = torch.device('cuda:7' if torch.cuda.is_available() else 'cpu')  # 使用方法.to(device, non_blocking=True) 
print(device)

In [ ]:
# 环境诊断
print(torch.__version__)            # 确认torch版本
print(torch.cuda.is_available())    # 必须返回True
print(torch.cuda.get_device_name(0))# 应识别到3090

In [ ]:
# 1. 组合模型实现
class GAT_KAN(nn.Module):
    def __init__(self
                 , gat_hyper_params
                 , kan_hyper_params
                 , kan_save_act=False
                 , device='cuda'):
        super().__init__()
        
        # 初始化 GAT 层容器
        self.gat_layers = nn.ModuleList()  # 存储多个GAT层的容器
        self.batch_norms = nn.ModuleList()  # 存储BN层的容器
        
        # 输入特征维度（原子周期表序号 + 原子电荷）
        in_channels = 2
        
        # 构建隐藏层（贝叶斯优化决定层数和每层维度）
        for out_channels in gat_hyper_params['hidden_dims']:
            # 添加 GAT 层：多头注意力机制实现
            self.gat_layers.append(
                GATConv(
                    in_channels,            # 输入维度，每添加一层，in_channels更新一次与上一层的输出维度相匹配
                    out_channels,           # 输出维度（每个注意力头的维度）
                    heads=gat_hyper_params['heads']  # 注意力头数量
                )
            )
            
            # 添加对应的BatchNorm层
            self.batch_norms.append(
                BatchNorm(out_channels * gat_hyper_params['heads'])  # BN的输入维度与GAT输出维度一致
            )
            
            # 更新输入维度：多头注意力的输出维度 = 头数 × 每头维度
            # 例如：当 heads=4, out_channels=32 → 实际输出 4×32=128 维
            in_channels = out_channels * gat_hyper_params['heads']
        
        # 最终输出层配置
        self.gat_out = gat_hyper_params['out_dim']  # 输出维度（贝叶斯优化决定）
        
        # 添加最终输出层（使用单注意力头）
        self.gat_layers.append(GATConv(in_channels     # 输入来自最后一层隐藏层
                                       , self.gat_out  # 输出目标维度
                                       , heads=1       # 最终层只用单头注意力
                                      ))

        self.batch_norms.append(BatchNorm(self.gat_out))   # 输出层的BN
        
        # KAN的输入维度计算
        kan_input_dim = self.gat_out + 7  # 确保12与experiment_condition_tensor维度匹配

        # KAN的width结构
        kan_width = [kan_input_dim]  # 输入层
        for dim in kan_hyper_params['hidden_dims']:
            if isinstance(dim, list) and len(dim) == 2:
                kan_width.append(dim)  # 保持[加法节点数, 乘法节点数]结构
            else:
                kan_width.append([dim, 0])  # 普通层用0表示无乘法节点
        kan_width.append(1)  # 输出层

        # mult_arity结构
        kan_mult_arity = [tuple()]  # 输入层无乘法节点
        for arity in kan_hyper_params['mult_arity']:
            kan_mult_arity.append(arity)  # 保持每层的元数组
        kan_mult_arity.append(tuple())  # 输出层无乘法节点

        self.kan = MultKAN(width=kan_width
                           , mult_arity=kan_mult_arity
                           , k=kan_hyper_params['k']
                           , grid=kan_hyper_params['grid']
                           , seed=666
                           , save_act=kan_save_act
                           , device=device
                          )

    def forward(self, x, edge_index, batch, extra_features):
        # GAT处理
        for gat_layer, bn_layer in zip(self.gat_layers[:-1], self.batch_norms[:-1]):
            x = gat_layer(x, edge_index) # GAT计算
            x = bn_layer(x)              # BN归一化
            x = F.elu(x)                 # 激活函数
            x = F.dropout(x, p=0.5, training=self.training) # dropout
        x = self.gat_layers[-1](x, edge_index)
        x = self.batch_norms[-1](x)
        
        # 图池化
        pooled = global_mean_pool(x, batch)
        
        # 特征拼接
        combined = torch.cat([pooled, extra_features], dim=1)
        
        # KAN处理
        return self.kan(combined), combined

In [ ]:
# 加载 mol2/chg 生成的 PyG 数据集。
from pathlib import Path
# Run from the repository root or this notebook's directory.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'environment.yml').is_file() else Path.cwd().parent
DATASET_PATH = PROJECT_ROOT / '02_datasets' / 'kare_gat_dataset.pt'
allset = torch.load(DATASET_PATH, map_location=device)

required_fields = ['name', 'conformer_id', 'experiment_condition_tensor', 'extraction_rate', 'atom_types_chg', 'edge_index']
missing_fields = [field for field in required_fields if not hasattr(allset[0], field)]
if missing_fields:
    raise AttributeError(f"数据集中缺少必要字段: {missing_fields}")

molecule_names = sorted({data.name for data in allset})
print(f"总样本数: {len(allset)}")
print(f"分子名数量: {len(molecule_names)}")
print(f"分子名: {molecule_names}")

In [ ]:
print(len(allset))
allset[0].experiment_condition_tensor

In [ ]:
# 数据增强前搜索的参数，对应的学习率0.004473146585817316
# GAT参数
gat_hyper_params = {'hidden_dims': [64, 96, 16]
                    , 'heads': 4
                    , 'out_dim': 5
                    }

# KAN参数
kan_hidden_dims = [10, [9, 6]]   # [10, [9, 6]]其中第二层[9, 6]是有乘法的部分
kan_mult_arity = [tuple(), (9, 3, 10, 6, 7, 3)]   # [tuple(), (9, 3, 10, 6, 7, 3)]， tuple()是给加法用的，(9, 3, 10, 6, 7, 3)是第二层乘法部分的参数

kan_hyper_params = {'hidden_dims': kan_hidden_dims
                    , 'mult_arity': kan_mult_arity
                    , 'k': 5
                    , 'grid': 4
                    }

In [ ]:
# # 数据增强搜索的参数，对应的学习率0.004888254764696056
# # GAT参数
# gat_hyper_params = {'hidden_dims': [96, 96, 96]
#                     , 'heads': 3
#                     , 'out_dim': 4
#                     }

# # KAN参数
# kan_hidden_dims = [10, 7]   # [10, [9, 6]]其中第二层[9, 6]是有乘法的部分
# kan_mult_arity = [tuple(), tuple()]   # [tuple(), (9, 3, 10, 6, 7, 3)]， tuple()是给加法用的，(9, 3, 10, 6, 7, 3)是第二层乘法部分的参数

# kan_hyper_params = {'hidden_dims': kan_hidden_dims
#                     , 'mult_arity': kan_mult_arity
#                     , 'k': 8
#                     , 'grid': 7
#                     }

In [ ]:
import os
import numpy as np
import copy
import pandas as pd
import torch
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import matplotlib
from torch_geometric.loader import DataLoader
from collections import defaultdict
import re


# 设置随机种子保证可重复性
RANDOM_SEED = 999
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# 兼容当前 pykan028 环境中的 matplotlib/IPython 版本组合，避免 plt.figure() 访问 rcParams._get 时报错。
if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = matplotlib.rcParams.get

# 基础路径设置
base_dir = PROJECT_ROOT / '04_cross_validation' / 'lomo_results'
os.makedirs(base_dir, exist_ok=True)

def get_molecule_name(data):
    return data.name


def sanitize_folder_name(name):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(name)).strip('_')


def make_leave_one_molecule_folds(dataset):
    molecule_to_indices = defaultdict(list)

    for idx, data in enumerate(dataset):
        molecule_to_indices[get_molecule_name(data)].append(idx)

    folds = []
    for molecule_name in sorted(molecule_to_indices):
        indices = molecule_to_indices[molecule_name]
        conformer_ids = sorted({dataset[idx].conformer_id for idx in indices})
        folds.append({
            'name': molecule_name,
            'indices': indices,
            'conformer_ids': conformer_ids,
            'size': len(indices)
        })

    return folds


def check_molecule_leakage(train_data, val_data):
    train_names = {get_molecule_name(data) for data in train_data}
    val_names = {get_molecule_name(data) for data in val_data}

    assert train_names.isdisjoint(val_names), f"训练集和验证集分子名泄漏: {train_names & val_names}"

    return len(train_names), len(val_names)


def fit_standardization_scalers(train_data):
    eps = 1e-8
    atom_features = torch.cat([data.atom_types_chg.float() for data in train_data], dim=0)
    condition_features = torch.cat([data.experiment_condition_tensor.float() for data in train_data], dim=0)
    targets = torch.cat([data.extraction_rate.float().view(-1, 1) for data in train_data], dim=0)

    scalers = {
        'atom_mean': atom_features.mean(dim=0, keepdim=True),
        'atom_std': atom_features.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps),
        'condition_mean': condition_features.mean(dim=0, keepdim=True),
        'condition_std': condition_features.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps),
        'target_mean': targets.mean(dim=0, keepdim=True),
        'target_std': targets.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    }
    return scalers


def apply_standardization(dataset, scalers):
    standardized_dataset = []
    for data in dataset:
        data_scaled = data.clone()
        data_scaled.atom_types_chg_scaled = (data.atom_types_chg.float() - scalers['atom_mean']) / scalers['atom_std']
        data_scaled.experiment_condition_tensor_scaled = (
            data.experiment_condition_tensor.float() - scalers['condition_mean']
        ) / scalers['condition_std']
        data_scaled.extraction_rate_scaled = (
            data.extraction_rate.float() - scalers['target_mean']
        ) / scalers['target_std']
        standardized_dataset.append(data_scaled)
    return standardized_dataset


def inverse_standardize_target(values, scalers):
    return values * scalers['target_std'].detach().cpu().numpy() + scalers['target_mean'].detach().cpu().numpy()


def set_adaptive_loss_ylim(train_losses, val_losses, lower_floor=0.0):
    losses = np.array(list(train_losses) + list(val_losses), dtype=float)
    losses = losses[np.isfinite(losses)]
    if losses.size == 0:
        return

    y_min = min(lower_floor, losses.min())
    y_max = losses.max()
    if y_max == y_min:
        padding = max(abs(y_max) * 0.05, 1.0)
    else:
        padding = (y_max - y_min) * 0.05
    plt.ylim(y_min - padding, y_max + padding)


# 设置不进入留一验证全程的分子名。这里填入的分子不会作为验证集，也不会进入任何一折训练集。
EXCLUDED_MOLECULE_NAMES = ['C12DGAA', 'DOAA', 'HDEHDGA', 
                           'Cyanex272', 'HEDEAP', 'HEHAMP', 'P204', 'P507', 'HEHHAP']
# EXCLUDED_MOLECULE_NAMES = []
excluded_molecule_names = set(EXCLUDED_MOLECULE_NAMES)
available_molecule_names = {get_molecule_name(data) for data in allset}
missing_excluded_molecule_names = sorted(excluded_molecule_names - available_molecule_names)
if missing_excluded_molecule_names:
    raise ValueError(f"排除名单中的分子名不在数据集中: {missing_excluded_molecule_names}")

allset_for_lomo = [
    data for data in allset
    if get_molecule_name(data) not in excluded_molecule_names
]
excluded_sample_count = len(allset) - len(allset_for_lomo)
print(f"排除分子名: {sorted(excluded_molecule_names)}")
print(f"排除样本数: {excluded_sample_count}")
print(f"进入LOMO的样本数: {len(allset_for_lomo)}")
print(f"进入LOMO的分子名数量: {len(available_molecule_names - excluded_molecule_names)}")

# 按分子名留一验证：每轮验证集包含同一 name 下的全部构象，conformer_id 不参与拆分。
folds = make_leave_one_molecule_folds(allset_for_lomo)

for fold, val_fold in enumerate(folds):
    val_molecule_name = sanitize_folder_name(val_fold['name'])
    fold_dir = os.path.join(base_dir, f"fold_{fold+1}_{val_molecule_name}")
    os.makedirs(fold_dir, exist_ok=True)

    val_indices = set(val_fold['indices'])
    train_data_raw = [data for idx, data in enumerate(allset_for_lomo) if idx not in val_indices]
    val_data = [allset_for_lomo[idx] for idx in val_fold['indices']]

    print(f"""
第{fold+1}折
原始数据划分：训练集{len(train_data_raw)}   验证集{len(val_data)}
===============================================================""")

    scalers = fit_standardization_scalers(train_data_raw)
    train_data = apply_standardization(train_data_raw, scalers)
    val_data = apply_standardization(val_data, scalers)

    print('标准化参数仅由当前折训练集计算，已应用到训练集和验证集')

    train_molecule_count, val_molecule_count = check_molecule_leakage(train_data, val_data)

    print(f'留一分子名：{val_fold["name"]}')
    print(f'验证集构象ID：{val_fold["conformer_ids"]}')
    print(f'最终划分：训练集{len(train_data)}   验证集{len(val_data)}')
    print(f'分子名数量：训练集{train_molecule_count}   验证集{val_molecule_count}')
    print(f"验证集分子名：{val_fold['name']}")
    
    # 保存数据集
    torch.save(train_data, os.path.join(fold_dir, f"train_set_fold{fold+1}.pt"))
    torch.save(val_data, os.path.join(fold_dir, f"val_set_fold{fold+1}.pt"))
    
    # 创建数据加载器
    train_loader = DataLoader(train_data, 
                             batch_size=len(train_data), 
                              # batch_size=1119, 
                             shuffle=True)
    val_loader = DataLoader(val_data, 
                           batch_size=len(val_data), 
                           shuffle=False)

    # 初始化模型
    model = GAT_KAN(gat_hyper_params
                    , kan_hyper_params
                    , kan_save_act=False
                    , device=device).to(device)

    # ========================开始第一阶段训练==========================
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.004473146585817316)  # 搜索的最佳学习率0.004473146585817316
    
    # def lr_lambda(epoch):
    #     if epoch < 700:
    #         return 1.0  # 保持原始学习率
    #     else:
    #         return 0.0001 / 0.004473146585817316  # 调整学习率为 0.0001
            
    # scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    criterion = nn.MSELoss().to(device)
    
    # 训练参数
    num_epochs = 500
    first_stage_best_epoch_range = (300, 500)  # 第一阶段只在这个闭区间内选择最佳模型，可自行修改
    first_stage_best_epoch_start, first_stage_best_epoch_end = first_stage_best_epoch_range
    if first_stage_best_epoch_start > first_stage_best_epoch_end:
        raise ValueError(f"第一阶段最佳模型epoch范围设置错误: {first_stage_best_epoch_range}")
    best_val_loss = float('inf')
    best_epoch = None
    best_model_info = None
    train_loss_list = []  # 初始化训练损失列表
    val_loss_list = []    # 初始化验证损失列表

    # 训练循环0-700
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for data in train_loader:
            # AdamW用的训练过程
            optimizer.zero_grad()
            output1, train_combined = model(data.atom_types_chg_scaled
                            , data.edge_index
                            , data.batch
                            , data.experiment_condition_tensor_scaled)
    
            # 计算损失
            loss = criterion(output1, data.extraction_rate_scaled.float())  # 预测标准化后的萃取率
            # loss = criterion(output1, data.logD.float())  # 预测logD
            # print(f'训练损失:{loss}')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss = train_loss + loss.item()
        train_loss = train_loss / len(train_loader)
        train_loss_list.append(train_loss)  # 记录训练损失
    
        # 验证
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for val_data in val_loader:
                # val_data.to(device)
                output2, val_combined = model(val_data.atom_types_chg_scaled
                                , val_data.edge_index
                                , val_data.batch
                                , val_data.experiment_condition_tensor_scaled)
                
                # 计算损失
                loss2 = criterion(output2, val_data.extraction_rate_scaled.float())  # 预测标准化后的萃取率
                # loss2 = criterion(output2, val_data.logD.float())  # 预测logD
                # print(f'验证损失:{loss2}', '\n')
                val_loss = val_loss + loss2.item()        
        val_loss = val_loss / len(val_loader)
        val_loss_list.append(val_loss)  # 记录验证损失
        # scheduler.step()  # 更新学习率（每个 epoch 后）
        
        # 只在指定epoch闭区间内选择第一阶段最佳模型
        if first_stage_best_epoch_start <= epoch <= first_stage_best_epoch_end and val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_model_info = {
                'epoch': best_epoch,
                'model_state_dict': copy.deepcopy(model.state_dict()),
                'gat_hyper_params': gat_hyper_params,
                'kan_hyper_params': kan_hyper_params,
                'learning_rate': optimizer.param_groups[0]['lr'],
                'val_loss': best_val_loss,
                'scalers': scalers
            }

    # =====================绘图全局样式设置=========================
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Times New Roman']
    plt.rcParams.update({
        'font.weight': 'bold',
        'axes.labelweight': 'bold',
        'axes.titleweight': 'bold',
        'axes.linewidth': 2.5,
        'font.size': 24,
        'axes.titlesize': 18,
        'axes.labelsize': 18,
        'xtick.labelsize': 16,
        'ytick.labelsize': 16,
        'legend.fontsize': 16
    })
    
    # =================== 新增：保存损失曲线数据 ===================
    loss_df = pd.DataFrame({
    'Epoch': range(len(train_loss_list)),
    'Train Loss': train_loss_list,
    'Validation Loss': val_loss_list
    })
    loss_df.to_excel(os.path.join(fold_dir, f"loss_curves_fold{fold+1}-0-{num_epochs}-best_epoch{best_epoch}.xlsx"), index=False)
    
    # =================== 新增：绘制损失曲线 ===================
    plt.figure(figsize=(5, 5))
    plt.plot(train_loss_list, label='Train Loss', color='#6A5ACD', linewidth=2)
    plt.plot(val_loss_list, label='Validation Loss', color='#CD5555', linewidth=2)
    plt.xlabel('Epoch', fontweight='bold')
    plt.ylabel('MSE Loss', fontweight='bold')
    plt.title(f'Fold {fold+1} Training Dynamics (Epoch 0-{num_epochs})', fontweight='bold')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    set_adaptive_loss_ylim(train_loss_list, val_loss_list)
    plt.tight_layout()
    plt.savefig(os.path.join(fold_dir, f'loss_curves_fold{fold+1}-0-{num_epochs}.svg'), 
                dpi=600, 
                bbox_inches="tight")
    plt.close()

    if best_model_info is None:
        raise ValueError(
            f"第一阶段最佳模型范围 {first_stage_best_epoch_range} 与实际训练epoch 0-{num_epochs - 1} 没有交集"
        )

    # 保存最佳模型
    torch.save(best_model_info, os.path.join(fold_dir, f"best_model_fold{fold+1}-0-{num_epochs}-best_epoch{best_epoch}.pt"))
    
    # ========================开始第二阶段训练==========================
    # 加载第一阶段训练的最佳模型进行二阶段训练
    model.load_state_dict(best_model_info['model_state_dict'])

    optimizer_2 = torch.optim.AdamW(model.parameters(), lr=0.0001)
    criterion_2 = nn.MSELoss().to(device)

    # 训练参数
    num_epochs_2 = 50
    best_val_loss_2 = float('inf')
    best_epoch_2 = 0
    best_model_info_2 = None
    train_loss_list_2 = []  # 初始化训练损失列表
    val_loss_list_2 = []    # 初始化验证损失列表

    # 训练循环700-1000
    for epoch in range(num_epochs_2):
        model.train()
        train_loss = 0
        for data in train_loader:
            # AdamW用的训练过程
            optimizer_2.zero_grad()
            output1, train_combined = model(data.atom_types_chg_scaled
                            , data.edge_index
                            , data.batch
                            , data.experiment_condition_tensor_scaled)
    
            # 计算损失
            loss = criterion_2(output1, data.extraction_rate_scaled.float())  # 预测标准化后的萃取率
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_2.step()
            train_loss = train_loss + loss.item()
        train_loss = train_loss / len(train_loader)
        train_loss_list_2.append(train_loss)  # 记录训练损失
    
        # 验证
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for val_data in val_loader:
                output2, val_combined = model(val_data.atom_types_chg_scaled
                                , val_data.edge_index
                                , val_data.batch
                                , val_data.experiment_condition_tensor_scaled)
                
                # 计算损失
                loss2 = criterion_2(output2, val_data.extraction_rate_scaled.float())  # 预测标准化后的萃取率
                val_loss = val_loss + loss2.item()        
        val_loss = val_loss / len(val_loader)
        val_loss_list_2.append(val_loss)  # 记录验证损失
        # scheduler.step()  # 更新学习率（每个 epoch 后）
        
        # 保存最佳模型
        if val_loss < best_val_loss_2:
            best_val_loss_2 = val_loss
            best_epoch_2 = epoch
            best_model_info_2 = {
                'epoch': best_epoch_2+num_epochs,
                'model_state_dict': copy.deepcopy(model.state_dict()),
                'gat_hyper_params': gat_hyper_params,
                'kan_hyper_params': kan_hyper_params,
                'learning_rate': optimizer_2.param_groups[0]['lr'],
                'val_loss': best_val_loss_2,
                'scalers': scalers
            }
    
    # =================== 新增：保存损失曲线数据 ===================
    loss_df = pd.DataFrame({
    'Epoch': range(len(train_loss_list_2)),
    'Train Loss': train_loss_list_2,
    'Validation Loss': val_loss_list_2
    })
    loss_df.to_excel(os.path.join(fold_dir, f"loss_curves_fold{fold+1}-{num_epochs}-{num_epochs+num_epochs_2}.xlsx"), index=False)
    
    # =================== 新增：绘制损失曲线 ===================
    plt.figure(figsize=(5, 5))
    plt.plot(train_loss_list_2, label='Train Loss', color='#6A5ACD', linewidth=2)
    plt.plot(val_loss_list_2, label='Validation Loss', color='#CD5555', linewidth=2)
    plt.xlabel('Epoch', fontweight='bold')
    plt.ylabel('MSE Loss', fontweight='bold')
    plt.title(f'Fold {fold+1} Training Dynamics (Epoch {num_epochs}-{num_epochs+num_epochs_2})', fontweight='bold')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    set_adaptive_loss_ylim(train_loss_list_2, val_loss_list_2)
    plt.tight_layout()
    plt.savefig(os.path.join(fold_dir, f'loss_curves_fold{fold+1}-{num_epochs}-{num_epochs+num_epochs_2}.svg'), 
                dpi=600, 
                bbox_inches="tight")
    plt.close()

    # 保存最佳模型
    torch.save(best_model_info_2, os.path.join(fold_dir, f"best_model_fold{fold+1}-{num_epochs}-{num_epochs+num_epochs_2}-best_epoch{best_epoch_2}.pt"))

    # 加载第二阶段训练的最佳模型进行验证集评估
    model.load_state_dict(best_model_info_2['model_state_dict'])
    
    # 获取各数据集预测结果
    def get_predictions(loader):
        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for data in loader:
                pred, _ = model(data.atom_types_chg_scaled, data.edge_index, data.batch, data.experiment_condition_tensor_scaled)
                all_preds.append(inverse_standardize_target(pred.cpu().numpy(), scalers))
                all_targets.append(data.extraction_rate.float().cpu().numpy())
        return np.concatenate(all_preds), np.concatenate(all_targets)
    
    # 绘制训练集和验证集结果图
    plt.figure(figsize=(5, 5))
    
    # 定义颜色
    train_color = '#6A5ACD'  # 蓝色
    val_color = '#EEB4B4'    # 绿色
    
    train_preds, train_targets = get_predictions(train_loader)
    val_preds, val_targets = get_predictions(val_loader)
    
    # 绘制训练集散点
    plt.scatter(train_targets, train_preds, color=train_color, alpha=0.6, label=f'Train R²: {r2_score(train_targets, train_preds):.4f}')
    # 绘制验证集散点
    plt.scatter(val_targets, val_preds, color=val_color, alpha=0.6, label=f'Validation R²: {r2_score(val_targets, val_preds):.4f}')
    # 绘制对角线
    min_value = min(train_targets.min(), val_targets.min())
    max_value = max(train_targets.max(), val_targets.max())
    plt.plot([min_value, max_value], [min_value, max_value], 'r--', linewidth=2)
    plt.xlabel('True Values')
    plt.ylabel('Predictions')
    plt.title('Train & Validation')
    plt.legend(loc='lower right')
    
    # 布局调整
    plt.tight_layout()
    plt.savefig(os.path.join(fold_dir, f'performance_train_val_fold{fold+1}-{num_epochs}-{num_epochs+num_epochs_2}-best_epoch{best_epoch_2}.svg')
                , dpi=600
                , bbox_inches="tight")
    plt.close()